In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import time
import json

import torch
import torch.nn as nn
import torchvision.transforms as transforms

from torchvision import models
from torch.utils.data import Dataset, DataLoader
from PIL import Image

print("PyTorch version:", torch.__version__)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
Device: cuda
GPU: Tesla T4


In [ ]:
DATASET_ROOT = "/content/drive/MyDrive/Dataset/plantwild"

DRIVE_MODEL_DIR = "/content/drive/MyDrive/PlantWild_Models"

os.makedirs(
    DRIVE_MODEL_DIR,
    exist_ok=True
)

print("Dataset path:")
print(DATASET_ROOT)

print("\nModel storage:")
print(DRIVE_MODEL_DIR)

Dataset path:
/content/drive/MyDrive/Dataset/plantwild

Model storage:
/content/drive/MyDrive/PlantWild_Models


In [ ]:
trainval_file = os.path.join(
    DATASET_ROOT,
    "trainval.txt"
)

classes_file = os.path.join(
    DATASET_ROOT,
    "classes.txt"
)

images_folder = os.path.join(
    DATASET_ROOT,
    "images"
)

print(
    "Dataset exists:",
    os.path.exists(DATASET_ROOT)
)

print(
    "trainval.txt exists:",
    os.path.exists(trainval_file)
)

print(
    "classes.txt exists:",
    os.path.exists(classes_file)
)

print(
    "images folder exists:",
    os.path.exists(images_folder)
)

Dataset exists: True
trainval.txt exists: True
classes.txt exists: True
images folder exists: True


In [ ]:
from collections import Counter

split_counts = Counter()

with open(
    trainval_file,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        image_path, class_id, mode = line.rsplit("=", 2)

        split_counts[int(mode)] += 1


print("Dataset split counts:\n")

for mode, count in sorted(split_counts.items()):

    print(
        f"Mode {mode}: {count} images"
    )

Dataset split counts:

Mode 0: 3677 images
Mode 1: 13045 images
Mode 2: 1820 images


In [ ]:
CLASS_NAMES = []

with open(
    classes_file,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        class_id, class_name = line.split(
            " ",
            1
        )

        CLASS_NAMES.append(
            class_name
        )


NUM_CLASSES = len(CLASS_NAMES)

print(
    f"Number of classes: {NUM_CLASSES}"
)

print("\nFirst 10 classes:")

for i, name in enumerate(
    CLASS_NAMES[:10]
):

    print(
        i,
        "→",
        name
    )

Number of classes: 89

First 10 classes:
0 → apple black rot
1 → apple leaf
2 → apple mosaic virus
3 → apple rust
4 → apple scab
5 → banana leaf
6 → banana panama disease
7 → basil downy mildew
8 → basil leaf
9 → bean halo blight


In [ ]:
transform_pipeline = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

In [ ]:
class PlantWildDataset(Dataset):

    def __init__(self, split_mode):

        self.samples = []

        with open(
            trainval_file,
            "r",
            encoding="utf-8"
        ) as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                image_path, class_id, mode = line.rsplit(
                    "=",
                    2
                )

                if int(mode) == split_mode:

                    full_path = os.path.join(
                        images_folder,
                        image_path
                    )

                    self.samples.append(
                        (
                            full_path,
                            int(class_id)
                        )
                    )

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, index):

        image_path, label = self.samples[index]

        image = Image.open(
            image_path
        ).convert("RGB")

        image = transform_pipeline(
            image
        )

        return image, label

In [ ]:
train_dataset = PlantWildDataset(
    split_mode=1
)

val_dataset = PlantWildDataset(
    split_mode=2
)

print(
    "Training images:",
    len(train_dataset)
)

print(
    "Validation images:",
    len(val_dataset)
)

Training images: 3677
Validation images: 13045


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Training DataLoader ready.")
print("Validation DataLoader ready.")

Training DataLoader ready.
Validation DataLoader ready.


In [ ]:
def create_mobilenet_model(
    num_classes=NUM_CLASSES
):

    model = models.mobilenet_v3_large(
        weights=models.MobileNet_V3_Large_Weights.DEFAULT
    )

    in_features = model.classifier[3].in_features

    model.classifier[3] = nn.Sequential(

        nn.Linear(
            in_features,
            512
        ),

        nn.Hardswish(),

        nn.Dropout(
            p=0.2
        ),

        nn.Linear(
            512,
            num_classes
        )
    )

    return model


model = create_mobilenet_model().to(device)

print(
    f"MobileNetV3-Large initialized."
)

print(
    f"Output classes: {NUM_CLASSES}"
)

print(
    f"Device: {device}"
)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 165MB/s]


MobileNetV3-Large initialized.
Output classes: 89
Device: cuda


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0001,
    weight_decay=0.0001
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

print(
    "Loss:",
    criterion
)

print(
    "Optimizer:",
    optimizer.__class__.__name__
)

print(
    "Scheduler:",
    scheduler.__class__.__name__
)

Loss: CrossEntropyLoss()
Optimizer: AdamW
Scheduler: ReduceLROnPlateau


In [ ]:
NUM_EPOCHS = 3

best_val_accuracy = 0.0

training_history = []

best_model_path = os.path.join(
    DRIVE_MODEL_DIR,
    "best_model.pth"
)

history_path = os.path.join(
    DRIVE_MODEL_DIR,
    "training_history.json"
)

print(
    "Number of epochs:",
    NUM_EPOCHS
)

print(
    "Best model will be saved to:"
)

print(
    best_model_path
)

Number of epochs: 3
Best model will be saved to:
/content/drive/MyDrive/PlantWild_Models/best_model.pth


In [ ]:
print("\n" + "=" * 70)
print("STARTING PLANTWILD TRAINING")
print("=" * 70)

print(
    f"Training images:   {len(train_dataset)}"
)

print(
    f"Validation images: {len(val_dataset)}"
)

print(
    "Batch size:        32"
)

print(
    f"Epochs:            {NUM_EPOCHS}"
)

print(
    f"Device:            {device}"
)


for epoch in range(NUM_EPOCHS):

    epoch_start = time.time()

    # ==========================================================
    # TRAINING
    # ==========================================================

    model.train()

    running_loss = 0.0
    total_correct = 0
    total_images = 0
    total_trained = 0
    next_train_report = 100

    print("\n" + "-" * 70)
    print(
        f"EPOCH {epoch + 1}/{NUM_EPOCHS} - TRAINING"
    )
    print("-" * 70)

    for images, labels in train_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        batch_size_actual = labels.size(0)

        running_loss += (
            loss.item()
            * batch_size_actual
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        total_correct += (
            predictions == labels
        ).sum().item()

        total_images += batch_size_actual

        total_trained += batch_size_actual

        while total_trained >= next_train_report:

            elapsed = (
                time.time()
                - epoch_start
            )

            current_loss = (
                running_loss
                / total_images
            )

            current_accuracy = (
                total_correct
                / total_images
            )

            print(
                f"✅ {next_train_report} "
                f"images trained "
                f"— Loss: {current_loss:.4f} "
                f"— Accuracy: {current_accuracy:.4f} "
                f"— {elapsed:.2f}s"
            )

            next_train_report += 100


    train_loss = (
        running_loss
        / total_images
    )

    train_accuracy = (
        total_correct
        / total_images
    )


    # ==========================================================
    # VALIDATION
    # ==========================================================

    print("\n🔎 Starting validation...")

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    total_validated = 0
    next_val_report = 100

    validation_start = time.time()

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            batch_size_actual = labels.size(0)

            val_loss += (
                loss.item()
                * batch_size_actual
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += batch_size_actual

            total_validated += batch_size_actual

            while total_validated >= next_val_report:

                print(
                    f"🔎 {next_val_report} "
                    f"validation images evaluated"
                )

                next_val_report += 100


    val_loss = (
        val_loss
        / val_total
    )

    val_accuracy = (
        val_correct
        / val_total
    )

    validation_time = (
        time.time()
        - validation_start
    )

    epoch_time = (
        time.time()
        - epoch_start
    )


    # ==========================================================
    # LEARNING RATE
    # ==========================================================

    scheduler.step(
        val_accuracy
    )

    current_lr = (
        optimizer.param_groups[0]["lr"]
    )


    # ==========================================================
    # RESULTS
    # ==========================================================

    print("\n" + "=" * 70)
    print(
        f"EPOCH {epoch + 1} RESULTS"
    )
    print("=" * 70)

    print(
        f"Training Loss:       {train_loss:.4f}"
    )

    print(
        f"Training Accuracy:   {train_accuracy:.4f}"
    )

    print(
        f"Validation Loss:     {val_loss:.4f}"
    )

    print(
        f"Validation Accuracy: {val_accuracy:.4f}"
    )

    print(
        f"Learning Rate:       {current_lr:.8f}"
    )

    print(
        f"Validation Time:     {validation_time:.2f} seconds"
    )

    print(
        f"Total Epoch Time:    {epoch_time:.2f} seconds"
    )


    # ==========================================================
    # SAVE HISTORY
    # ==========================================================

    epoch_record = {

        "epoch": epoch + 1,

        "train_loss": train_loss,

        "train_accuracy": train_accuracy,

        "val_loss": val_loss,

        "val_accuracy": val_accuracy,

        "learning_rate": current_lr,

        "epoch_time": epoch_time
    }

    training_history.append(
        epoch_record
    )


    with open(
        history_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            training_history,
            f,
            indent=4
        )


    # ==========================================================
    # SAVE BEST MODEL
    # ==========================================================

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            best_model_path
        )

        print("\n🏆 NEW BEST MODEL!")

        print(
            f"Best Validation Accuracy: "
            f"{best_val_accuracy:.4f}"
        )

        print(
            "Saved to:"
        )

        print(
            best_model_path
        )

    else:

        print(
            "\nBest Validation Accuracy remains: "
            f"{best_val_accuracy:.4f}"
        )


# ==============================================================
# TRAINING COMPLETE
# ==============================================================

print("\n" + "=" * 70)
print("🎉 TRAINING COMPLETE")
print("=" * 70)

print(
    f"Best Validation Accuracy: "
    f"{best_val_accuracy:.4f}"
)

print("\nBest model:")

print(
    best_model_path
)

print("\nTraining history:")

print(
    history_path
)


STARTING PLANTWILD TRAINING
Training images:   3677
Validation images: 13045
Batch size:        32
Epochs:            3
Device:            cuda

----------------------------------------------------------------------
EPOCH 1/3 - TRAINING
----------------------------------------------------------------------
✅ 100 images trained — Loss: 4.4932 — Accuracy: 0.0156 — 58.14s
✅ 200 images trained — Loss: 4.4888 — Accuracy: 0.0179 — 90.36s
✅ 300 images trained — Loss: 4.4859 — Accuracy: 0.0125 — 107.21s
✅ 400 images trained — Loss: 4.4817 — Accuracy: 0.0144 — 132.78s
✅ 500 images trained — Loss: 4.4772 — Accuracy: 0.0195 — 147.80s
✅ 600 images trained — Loss: 4.4755 — Accuracy: 0.0214 — 172.81s
✅ 700 images trained — Loss: 4.4743 — Accuracy: 0.0227 — 190.67s
✅ 800 images trained — Loss: 4.4719 — Accuracy: 0.0250 — 213.94s
✅ 900 images trained — Loss: 4.4672 — Accuracy: 0.0323 — 241.79s
✅ 1000 images trained — Loss: 4.4633 — Accuracy: 0.0361 — 256.01s
✅ 1100 images trained — Loss: 4.4602 — Acc

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 2. 
  warnings.warn(str(msg))


🔎 3000 validation images evaluated
🔎 3100 validation images evaluated
🔎 3200 validation images evaluated
🔎 3300 validation images evaluated
🔎 3400 validation images evaluated
🔎 3500 validation images evaluated
🔎 3600 validation images evaluated


KeyboardInterrupt: 

In [ ]:
print(
    "Best model exists:",
    os.path.exists(best_model_path)
)

if os.path.exists(best_model_path):

    size_mb = (
        os.path.getsize(best_model_path)
        / (1024 ** 2)
    )

    print(
        f"Model size: {size_mb:.2f} MB"
    )

print(
    "\nTraining history exists:",
    os.path.exists(history_path)
)

In [ ]:
test_dataset = PlantWildDataset(
    split_mode=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    "Test images:",
    len(test_dataset)
)

In [ ]:
model = create_mobilenet_model().to(device)

model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device
    )
)

model.eval()

print(
    "✅ Best model loaded successfully."
)

In [ ]:
test_correct = 0
test_total = 0

test_start = time.time()

print("\nStarting final test evaluation...\n")

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        outputs = model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        test_correct += (
            predictions == labels
        ).sum().item()

        test_total += labels.size(0)

        if test_total % 100 < 32:
            print(
                f"Evaluated {test_total} test images"
            )


test_accuracy = (
    test_correct
    / test_total
)

test_time = (
    time.time()
    - test_start
)

print("\n" + "=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)

print(
    f"Test images: {test_total}"
)

print(
    f"Correct:     {test_correct}"
)

print(
    f"Accuracy:    {test_accuracy:.4f}"
)

print(
    f"Accuracy %:  {test_accuracy * 100:.2f}%"
)

print(
    f"Test time:   {test_time:.2f} seconds"
)